In [ ]:
# Ambiente do notebook: garante a lib `model` no kernel que estiver rodando. Numa maquina
# nova, isto instala o projeto (editavel) com as dependencias de notebook do pyproject.
# Nao instala nada se `model` ja' importa, e NUNCA instala num interpretador fora de venv
# (o Python do ESP-IDF nao deve receber pacotes).
import subprocess
import sys
from pathlib import Path

RAIZ = next((p for p in [Path.cwd(), *Path.cwd().parents]
             if (p / "pyproject.toml").exists() and (p / "utils").is_dir()), None)
assert RAIZ is not None, "nao achei a raiz do repo (pyproject.toml + utils/) a partir do cwd"

# Onde o pacote mora (layout plano hoje; `src/` se ele voltar para la').
PACOTE = next((p for p in (RAIZ, RAIZ / "src") if (p / "model" / "__init__.py").exists()), None)
assert PACOTE is not None, f"nao achei model/__init__.py em {RAIZ} nem em {RAIZ / 'src'}"

try:
    import model
except ModuleNotFoundError:
    if sys.prefix == sys.base_prefix:
        raise RuntimeError(
            "este kernel NAO esta' dentro de um venv e `model` nao importa:\n"
            f"  kernel: {sys.executable}\n"
            "Escolha o kernel do projeto (`.venv`) em vez de instalar pacotes num Python de "
            "sistema/embutido: o Python do ESP-IDF nao deve receber pacotes.") from None
    print(f"instalando `model` (editavel) de {RAIZ} ...", flush=True)
    subprocess.run([sys.executable, "-m", "pip", "install", "-e", f"{RAIZ}[notebooks]"],
                   check=True)
    # Um .pth de editable so' e' lido no START do processo: num kernel que ja' estava
    # rodando, o import na mesma sessao precisa do caminho na mao (num kernel novo, o
    # .pth recem-instalado ja' basta).
    sys.path.insert(0, str(PACOTE))
    import model

print(f"model : {model.__file__}")
print(f"kernel: {sys.executable}")


> ## ⚠ Este notebook está defasado — decisão pendente
>
> Ele **não roda** contra o layout atual do projeto e não foi refatorado junto com os outros
> (2026-10-11). O que está errado, em ordem de gravidade:
>
> 1. **Lê arquivos que não existem:** `data/processed/{train,validation,test}.csv`. Depois da
>    reorganização, o split vive em `data/splited/` (`utils/dataset_split.ipynb`).
> 2. **Usa `ghi_cs`**, que está em `data/processed/*.csv` mas **não** nos CSVs do split — o split
>    não escreve essa coluna.
> 3. **Depende do `.venv-tf`** (TensorFlow), que não está no working tree. O TensorFlow não tem
>    wheel para o Python 3.14 do `.venv`.
> 4. Fala em `utils/server.ipynb` (hoje `model_teste.ipynb`) e em `MODEL_SAMPLES` do `main.cpp`
>    (hoje `kAxes = 7`, caminho RF).
>
> O caminho MLP (Keras → Edge Impulse) ficou **fora do escopo** da refatoração para a lib
> `src/model/` — decisão adiada com o autor. Enquanto ela não vier, trate este arquivo como
> histórico, não como pipeline.


# knob-senses — treino e exportação da regressão de PCA

Treina dois modelos para prever **PCA** (potência CA do inversor, em kW) a partir das
features de `data/processed/`, e **exporta o modelo Keras no formato que o
Edge Impulse aceita** (*Bring Your Own Model*).

**O alvo é `PCA` uma hora à frente.** Cada amostra vê `JANELA` horas consecutivas de
features e responde com a potência de `t+1`, não a da hora corrente.

## Papel de cada modelo

| modelo | para que serve |
|---|---|
| média do treino | piso. Qualquer modelo tem que bater isto |
| **Random Forest** | referência de qualidade. O Edge Impulse **não** aceita sklearn, então ele não é exportado — está aqui para dizer se a rede está deixando desempenho na mesa |
| **Keras (MLP)** | é o modelo exportado. Vira `.tflite` dentro do Edge Impulse |

## Os dados

`data/processed/{train,validation,test}.csv`, gerados por `utils/dataset_analisys.ipynb`:

- **features**: `temperatura_ambiente`, `ghi_cs`, `jacques`, `ghit`, `sin1`, `cos1`
- **alvo**: `PCA`, em kW, **sem normalizar** — a saída do modelo está na unidade física
- vêm em **unidade física**; a normalização é feita **aqui**, ajustada só no treino
- split **70/15/15 por tempo**, sem embaralhar

A série é **só diurna** (05h às 17h) e tem buracos — o apagão de jun/2022 e ~5% dos
slots intradiários faltando. Duas consequências que o código trata e que valem saber:

- às 17h o registro seguinte é 06h do dia seguinte, **13 horas depois**. Um `shift(1)`
  de linha não é "1 hora à frente"; o alvo tem que ser casado por `datetime + 1h`.
- uma janela que atravesse a noite não é histórico de verdade, e é descartada.

Como o corte é por tempo, validação e teste são **futuro de verdade**. Os números
abaixo são honestos, sem vazamento — e por isso são mais pessimistas que um split
aleatório seria.

> **Atenção:** o split de teste (2024-05-26 a 2024-12-31) cai inteiro dentro de um
> evento de degradação do sistema — a razão `PCA/ghit` cai ~14% a partir de ago/2024
> contra a média de 2021-2023. O MAE no teste mistura erro do modelo com essa perda,
> então ele é **pior que o erro típico em operação normal**. Ver a seção 6 de
> `utils/dataset_analisys.ipynb`.

## 1. Configuração

In [ ]:
from pathlib import Path
import json
import shutil
import time
import warnings
import zipfile

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import tensorflow as tf
import keras
from sklearn.dummy import DummyRegressor
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

RAIZ = Path.cwd().parent          # os notebooks moram em utils/
ENTRADA = RAIZ / "data" / "processed"
SAIDA = RAIZ / "data" / "modelo_exportado"

SEMENTE = 42
ALVO = "PCA"
FEATURES = ["temperatura_ambiente", "ghi_cs", "jacques", "ghit", "sin1", "cos1"]

# Janela de historico por amostra. O alvo e' PCA em t+1 (1 hora a frente).
#
# JANELA=2 medi melhor que 1, 3, 4, 6, 8 e 12. A janela entrega a TENDENCIA do
# GHI, que e' o unico sinal sobre a hora seguinte -- sem ela o modelo teria que
# extrapolar o clima sozinho. A partir de 3h nao melhora e so encarece o frame.
# Medido no teste: MAE 0,1930 (JANELA=1) -> 0,1496 (JANELA=2), media de 3 sementes.
#
# PCA defasado como entrada foi testado e NAO ajuda: empata sozinho e piora somado
# a janela. PCA(t) ja esta implicito no clima de t. Entao o server continua
# fatando o PCA fora e mandando so os 6 eixos.
JANELA = 2

# Ordem do achatamento: cronologica, mais antiga primeiro. E' a mesma que o
# ringLinearize() do firmware monta no ESP32 -- se mudar aqui, muda la.
NOMES = [f"{c}_t{j}" for j in range(JANELA - 1, -1, -1) for c in FEATURES]

# O inversor satura em ~2 kW e o dataset nao tem zero (o 0 era sentinela de
# faltante, removido na analise). Prever fora desta faixa e fisicamente impossivel.
PCA_MIN, PCA_MAX = 0.01, 2.02

keras.utils.set_random_seed(SEMENTE)

print("tensorflow", tf.__version__, "| keras", keras.__version__)
print("entrada:", ENTRADA)
print("saida  :", SAIDA)
print(f"janela : {JANELA} h x {len(FEATURES)} features = {len(NOMES)} entradas")

## 2. Carga e sanidade

In [ ]:
def carrega(nome):
    d = pd.read_csv(ENTRADA / f"{nome}.csv")
    faltando = [c for c in FEATURES + [ALVO] if c not in d.columns]
    assert not faltando, f"{nome}.csv sem as colunas {faltando}"
    return d.sort_values("datetime").reset_index(drop=True)


def monta(d):
    """X = JANELA horas de features (mais antiga primeiro), y = PCA em t+1.

    Duas armadilhas do dataset, tratadas aqui:

    - a serie e' so' diurna (05h..17h). As 17h o registro seguinte e' 06h do dia
      seguinte, 13h depois -- nao serve como alvo de 1h. Por isso o alvo e' casado
      por `datetime + 1h`, e nao por shift de linha.
    - a janela tem que ser contigua: um buraco de 13h no meio nao e' historico.
    """
    dt = pd.to_datetime(d["datetime"])

    por_instante = dict(zip(dt, d[ALVO]))
    alvo = (dt + pd.Timedelta(hours=1)).map(por_instante)

    gap = dt.diff().dt.total_seconds() / 3600
    contiguo = alvo.notna()
    for j in range(1, JANELA):
        contiguo &= (gap.shift(j - 1) == 1.0)

    origens = [(f"{c}_t{j}", j, c)
               for j in range(JANELA - 1, -1, -1) for c in FEATURES]
    X = pd.DataFrame({nome: d[c].shift(j) for nome, j, c in origens})[contiguo]
    assert list(X.columns) == NOMES, "ordem das colunas saiu da NOMES"
    assert not X.isna().any().any(), "janela com NaN: mascara de contiguidade errada"

    return X.to_numpy(np.float32), alvo[contiguo].to_numpy(np.float32)


treino, valid, teste = carrega("train"), carrega("validation"), carrega("test")

# Os csv vem em unidade FISICA (ghi_cs ~625 W/m2, temperatura ~27 C). A normalizacao
# e' feita aqui, e ajustada SO no treino: usar validacao ou teste contaminaria o
# modelo com a estatistica do futuro.
#
# Ela tambem e' o contrato com o firmware -- as constantes vao para normalizacao.json
# na secao 10, e o ESP32 precisa aplicar a MESMA conta antes de inferir.
MEDIA = treino[FEATURES].mean()
DESVIO = treino[FEATURES].std(ddof=0).replace(0.0, 1.0)   # coluna constante nao divide por zero

for d in (treino, valid, teste):
    d[FEATURES] = (d[FEATURES] - MEDIA) / DESVIO

X_tre, y_tre = monta(treino)
X_val, y_val = monta(valid)
X_tes, y_tes = monta(teste)

for nome, X, y in (("treino", X_tre, y_tre), ("validacao", X_val, y_val), ("teste", X_tes, y_tes)):
    print(f"{nome:10} {len(X):6} amostras de {X.shape[1]:2} entradas | "
          f"PCA em t+1 {y.min():.2f}..{y.max():.2f} kW | media {y.mean():.3f}")

# O corte por continuidade descarta ~1 linha por dia (as 17h) e as bordas dos
# splits. Se sobrar muito pouco, o casamento do alvo esta errado.
assert len(X_tes) > 1500, f"teste ficou com so {len(X_tes)} amostras"

# Confere a normalizacao recem-aplicada. Se cair, MEDIA/DESVIO sairam do lugar.
for c in FEATURES:
    assert abs(treino[c].mean()) < 1e-6, f"{c} nao esta centrada no treino"
    assert abs(treino[c].std(ddof=0) - 1) < 1e-6, f"{c} nao esta em escala unitaria no treino"
print("\nOK: features do treino em z-score (media 0, desvio 1)")

## 3. Métricas

Três números, em kW todos os três (R² é adimensional):

- **MAE** — o erro típico. É a métrica de cabeça: compare com a potência do inversor
  (~2 kW) para ter noção de escala.
- **RMSE** — igual ao MAE, mas pune erro grande com mais força.
- **R²** — fração da variância explicada.

O julgamento é **no split de teste**, que é o futuro. A validação só serve para
early stopping e para escolher hiperparâmetro; usá-la para reportar seria escolher a
dedo o resultado.

In [ ]:
def avalia(nome, y, pred):
    """MAE/RMSE em kW e R2 adimensional. Reporta tambem com clip na faixa fisica."""
    pred_clip = np.clip(pred, PCA_MIN, PCA_MAX)
    linha = {
        "modelo": nome,
        "MAE (kW)": mean_absolute_error(y, pred),
        "RMSE (kW)": float(np.sqrt(mean_squared_error(y, pred))),
        "R2": r2_score(y, pred),
        "MAE com clip (kW)": mean_absolute_error(y, pred_clip),
        "vies (kW)": float((pred - y).mean()),
    }
    print(f"{nome:20} MAE {linha['MAE (kW)']:.4f} kW | RMSE {linha['RMSE (kW)']:.4f} kW "
          f"| R2 {linha['R2']:.4f} | vies {linha['vies (kW)']:+.4f} kW")
    return linha


resultados = []

## 4. Piso: a média do treino

O modelo mais burro possível que ainda acerta a média. Se um modelo de verdade não
bater isto com folga, ele não aprendeu nada.

In [ ]:
media = DummyRegressor(strategy="mean").fit(X_tre, y_tre)
resultados.append(avalia("media do treino", y_tes, media.predict(X_tes).ravel()))

## 5. Random Forest — referência

`n_estimators=300` com `min_samples_leaf=2`. Não há busca de hiperparâmetro: o RF aqui
é régua, não candidato — o Edge Impulse não aceita modelo sklearn.

Se a rede Keras ficar muito atrás do RF, é sinal de que a arquitetura ou o treino
precisam de atenção. Se ficar perto, a rede está boa e é ela que vai para o firmware,
porque é a única que o Edge Impulse sabe compilar para o ESP32.

In [ ]:
rf = RandomForestRegressor(
    n_estimators=300,
    min_samples_leaf=2,
    n_jobs=-1,
    random_state=SEMENTE,
)

t0 = time.perf_counter()
rf.fit(X_tre, y_tre)
print(f"treino do RF: {time.perf_counter() - t0:.1f} s")

pred_rf = rf.predict(X_tes)
resultados.append(avalia("random forest", y_tes, pred_rf))

In [ ]:
importancia = pd.Series(rf.feature_importances_, index=NOMES).sort_values()

ax = importancia.plot.barh(figsize=(6.5, 4.6), color="#4c72b0")
ax.set_xlabel("importancia (reducao de impureza)")
ax.set_title(f"Random Forest - importancia das {len(NOMES)} entradas")
plt.tight_layout()
plt.show()

# Se _t1 (a hora anterior) nao aparecer entre as primeiras, a janela nao esta
# ajudando e JANELA=1 seria suficiente.
print(importancia.round(4).to_string())

## 6. Rede Keras — o modelo exportado

`12 -> 64 -> 32 -> 1` (ou `JANELA × 6 -> 64 -> 32 -> 1`), densa, ReLU nas escondidas e
**linear na saída**.

Decisões que valem registro:

- **Entrada achatada, não sequencial.** A janela vira um vetor de `JANELA × 6` na
  ordem cronológica. Um MLP denso não sabe que a entrada é uma série, então quem
  carrega a noção de tempo são os próprios pesos de cada hora. É deliberado: a camada
  recorrente (`LSTM`/`GRU`) não aparece na lista de operadores que o Edge Impulse
  compila para o ESP32.
- **Saída linear e em kW, sem normalizar o alvo.** A tentação é padronizar o alvo para
  estabilizar o MSE. Mas aí a rede passaria a prever z-score e o firmware teria que
  desfazer a conta (`y = y' * desvio + media`), o que é uma multiplicação a mais no
  ESP32 e um lugar a mais para errar. Como PCA vive em `[0, 2]`, o MSE cru já é
  bem-condicionado e não precisa de ajuda.
- **Rede pequena de propósito.** ~2,5 mil parâmetros. O Edge Impulse compila isto para
  o compilador EON / ESP32; rede maior aqui não compra acurácia e só encarece o binário.
- **Early stopping no `val_loss`**, com `restore_best_weights`. Paciência de 25 épocas,
  teto de 200.

In [ ]:
modelo = keras.Sequential(
    [
        keras.layers.Input(shape=(len(NOMES),), name="features"),
        keras.layers.Dense(64, activation="relu"),
        keras.layers.Dense(32, activation="relu"),
        keras.layers.Dense(1, name="pca_kw"),
    ],
    name="knob_senses_pca",
)

modelo.compile(optimizer=keras.optimizers.Adam(1e-3), loss="mse", metrics=["mae"])
modelo.summary()

In [ ]:
chamadas = [
    keras.callbacks.EarlyStopping(
        monitor="val_loss", patience=25, restore_best_weights=True, verbose=1
    ),
    keras.callbacks.ReduceLROnPlateau(
        monitor="val_loss", factor=0.5, patience=8, min_lr=1e-5, verbose=1
    ),
]

t0 = time.perf_counter()
historico = modelo.fit(
    X_tre,
    y_tre,
    validation_data=(X_val, y_val),
    epochs=200,
    batch_size=512,
    callbacks=chamadas,
    verbose=0,
)
print(f"treino da rede: {time.perf_counter() - t0:.1f} s")
print(f"parou na epoca {len(historico.history['loss'])} "
      f"(melhor val_loss {min(historico.history['val_loss']):.5f})")

In [ ]:
fig, (eixo_loss, eixo_mae) = plt.subplots(1, 2, figsize=(11, 3.4))

eixo_loss.plot(historico.history["loss"], label="treino")
eixo_loss.plot(historico.history["val_loss"], label="validacao")
eixo_loss.set_xlabel("epoca")
eixo_loss.set_ylabel("MSE (kW2)")
eixo_loss.set_title("loss")
eixo_loss.set_yscale("log")
eixo_loss.legend()

eixo_mae.plot(historico.history["mae"], label="treino")
eixo_mae.plot(historico.history["val_mae"], label="validacao")
eixo_mae.set_xlabel("epoca")
eixo_mae.set_ylabel("MAE (kW)")
eixo_mae.set_title("erro absoluto medio")
eixo_mae.legend()

plt.tight_layout()
plt.show()

In [ ]:
pred_keras = modelo.predict(X_tes, batch_size=512, verbose=0).ravel()
resultados.append(avalia("keras (MLP)", y_tes, pred_keras))

## 7. Comparação

In [ ]:
tabela = pd.DataFrame(resultados).set_index("modelo").round(4)
tabela

In [ ]:
def dispersao(eixo, y, pred, titulo):
    """Previsto contra real. A linha vermelha e a predicao perfeita; quanto mais
    perto dela, melhor. A cor e a contagem de pontos (escala log), porque 26 mil
    pontos em dispersao simples viram uma mancha opaca."""
    eixo.hexbin(y, pred, gridsize=55, bins="log", cmap="viridis", extent=(0, 2.1, 0, 2.1))
    eixo.plot([0, 2.1], [0, 2.1], "r--", lw=1)
    eixo.set_title(titulo)
    eixo.set_xlabel("PCA real (kW)")
    eixo.set_aspect("equal")


fig, eixos = plt.subplots(1, 3, figsize=(13.5, 4.2), sharey=True)
eixos[0].set_ylabel("PCA previsto (kW)")
dispersao(eixos[0], y_tes, media.predict(X_tes).ravel(), "media do treino")
dispersao(eixos[1], y_tes, pred_rf, "random forest")
dispersao(eixos[2], y_tes, pred_keras, "keras (MLP)")
plt.tight_layout()
plt.show()

In [ ]:
fig, eixos = plt.subplots(1, 2, figsize=(11, 3.6), sharey=True)

for eixo, nome, pred in (
    (eixos[0], "random forest", pred_rf),
    (eixos[1], "keras (MLP)", pred_keras),
):
    eixo.hexbin(pred, pred - y_tes, gridsize=55, bins="log", cmap="magma")
    eixo.axhline(0, color="r", ls="--", lw=1)
    eixo.set_xlabel("PCA previsto (kW)")
    eixo.set_title(f"residuo - {nome}")

eixos[0].set_ylabel("previsto - real (kW)")
plt.tight_layout()
plt.show()

## 8. Exportação para o Edge Impulse

Regras do *Bring Your Own Model* que restringem o que sai daqui:

- **formatos aceitos**: TensorFlow SavedModel (`.zip`), ONNX (`.onnx`) ou LiteRT
  (`.tflite`). **`.h5` não entra pelo Studio** — o caminho é o SavedModel.
- o `.zip` precisa conter um diretório `saved_model/` com o `saved_model.pb` dentro
- **1 tensor de entrada e 1 de saída**, ambos com dimensão de batch
- **o batch tem que ser 1**: entrada `(1, 12)` — `JANELA × FEATURES` — e para regressão
  a saída **tem que ser `(1, 1)`**, exatamente o formato desta rede
- a quantização int8 é opcional e só vale para upload em SavedModel/ONNX; ela pede um
  `.npy` em que **cada elemento tem a forma da entrada do modelo** — daí o
  `(n, 1, 12)`

O modelo treinado tem batch dinâmico (`None, 12`). A assinatura vira `(1, 12)` na
exportação, sem retreinar: nenhum peso muda.

In [ ]:
SAIDA.mkdir(parents=True, exist_ok=True)

assinatura = [tf.TensorSpec(shape=(1, len(NOMES)), dtype=tf.float32, name="features")]
dir_saved = SAIDA / "saved_model"
if dir_saved.exists():
    shutil.rmtree(dir_saved)
modelo.export(dir_saved, input_signature=assinatura, verbose=False)

# O zip guarda "saved_model/..." na raiz - e o que o Studio espera.
caminho_zip = SAIDA / "modelo_edge_impulse.zip"
with zipfile.ZipFile(caminho_zip, "w", zipfile.ZIP_DEFLATED) as z:
    for arquivo in sorted(dir_saved.rglob("*")):
        if arquivo.is_file():
            z.write(arquivo, arquivo.relative_to(SAIDA))

print(f"{caminho_zip.name}  {caminho_zip.stat().st_size / 1024:.1f} KB")
with zipfile.ZipFile(caminho_zip) as z:
    for nome in z.namelist():
        print("   ", nome)

In [ ]:
# .tflite: o Studio tambem aceita upload direto. Convertido A PARTIR DO SavedModel de
# batch=1, entao herda a assinatura (1, NOMES) -> (1,1) em vez de batch dinamico.
# Serve tambem para ver o tamanho que o firmware vai carregar.
conversor = tf.lite.TFLiteConverter.from_saved_model(str(dir_saved))
tflite_bytes = conversor.convert()
caminho_tflite = SAIDA / "modelo.tflite"
caminho_tflite.write_bytes(tflite_bytes)
print(f"{caminho_tflite.name}  {len(tflite_bytes) / 1024:.1f} KB")

# Dados representativos da quantizacao int8: cada elemento com a forma da entrada do
# modelo, (1, NOMES). Uma amostra da VALIDACAO - o teste nao entra aqui, senao a
# quantizacao veria o futuro.
amostra = X_val[:2000].reshape(-1, 1, len(NOMES)).astype(np.float32)
caminho_npy = SAIDA / "dados_representativos.npy"
np.save(caminho_npy, amostra)
print(f"{caminho_npy.name}  {amostra.shape}  {caminho_npy.stat().st_size / 1024:.1f} KB")

## 9. Conferência do artefato

Exportar não é o mesmo que exportar certo. Aqui o `.zip` é reaberto e as previsões
dele são comparadas com as do modelo que ainda está na memória. Se divergirem, o
artefato está quebrado e o erro apareceria só no ESP32.

In [ ]:
recarregado = tf.saved_model.load(str(dir_saved))
serving = recarregado.signatures["serving_default"]

print("assinatura de entrada:")
for nome, spec in serving.structured_input_signature[1].items():
    print(f"   {nome:12} shape={spec.shape} dtype={spec.dtype.name}")
print("assinatura de saida:")
for nome, spec in serving.structured_outputs.items():
    print(f"   {nome:12} shape={spec.shape} dtype={spec.dtype.name}")

# A entrada do SavedModel e keyword-only e o batch e fixo em 1. Entao a conferencia e
# linha a linha - que e exatamente como o ESP32 vai chamar. Lote maior so caberia
# reexportando com outra assinatura.
n = 64
do_saved = np.array([
    float(list(serving(features=tf.constant(x.reshape(1, -1))).values())[0].numpy().ravel()[0])
    for x in X_tes[:n]
])
do_keras = modelo(X_tes[:n], training=False).numpy().ravel()

diferenca = float(np.abs(do_saved - do_keras).max())
print(f"\nmaior diferenca entre o zip recarregado e o modelo original: "
      f"{diferenca:.2e} kW (em {n} linhas)")
assert diferenca < 1e-5, "o SavedModel exportado nao reproduz o modelo treinado"
print("OK: o zip reproduz o modelo treinado")

In [ ]:
# tf.lite.Interpreter esta marcado como obsoleto em favor do pacote ai_edge_litert.
# A conversao continua valida e o aviso nao ajuda em nada aqui, entao fica silenciado
# em vez de aparecer no meio dos numeros da conferencia.
with warnings.catch_warnings():
    warnings.filterwarnings("ignore", message=".*tf.lite.Interpreter is deprecated.*")
    interpretador = tf.lite.Interpreter(model_path=str(caminho_tflite))
    interpretador.allocate_tensors()
    entrada = interpretador.get_input_details()[0]
    saida = interpretador.get_output_details()[0]

    print("tflite entrada:", entrada["shape"], entrada["dtype"].__name__)
    print("tflite saida  :", saida["shape"], saida["dtype"].__name__)

    interpretador.set_tensor(entrada["index"], X_tes[:1].reshape(entrada["shape"]))
    interpretador.invoke()
    previsto_tflite = float(interpretador.get_tensor(saida["index"]).ravel()[0])

print(f"\nprimeira linha do teste:")
print(f"   real            {float(y_tes[0]):.4f} kW")
print(f"   keras           {float(pred_keras[0]):.4f} kW")
print(f"   tflite          {previsto_tflite:.4f} kW")

## 10. O que o firmware precisa saber

O modelo **não** recebe as grandezas físicas: recebe z-scores. Quem chama a inferência
no ESP32 tem que normalizar antes, com as constantes gravadas em
`normalizacao.json` (seção 9) — que são calculadas **só no split de treino**, nesta
execução. Não as transcreva para o firmware à mão: leia do arquivo, senão elas voltam
a ficar obsoletas na próxima vez que os csv forem regerados.

São **seis** pares média/desvio, um por feature de `FEATURES`.

### A janela tem que ser montada no formato certo

Cada inferência recebe `JANELA` amostras **consecutivas de 1 hora**, da mais antiga
para a mais recente, cada uma com as 6 features normalizadas. A saída é o PCA de
**1 hora depois** da amostra mais recente.

O `ringLinearize()` de `src/main.cpp` já monta exatamente essa ordem — mas hoje ele
está desligado: `MODEL_SAMPLES` vale 1. Para o modelo treinado aqui, precisa valer
`JANELA`.

### Três armadilhas

- **A série é só diurna e o alvo nunca é zero.** As amostras com `PCA == 0` foram
  removidas na análise por serem sentinela de dado faltante. Como as 17h não têm um
  `t+1` válido, o modelo **nunca é treinado a prever depois do fim do dia** — ele não
  vê a noite, e não deve ser chamado nela.
- **A saturação do inversor é um teto de verdade.** Há acúmulo de amostras coladas em
  ~2 kW; previsão acima disso é impossível e deve ser ceifada com o clip em
  `[PCA_MIN, PCA_MAX]`.
- **Horizonte de 1h, não "próxima amostra".** Se o firmware pular horas (retomada
  depois de uma queda, por exemplo), a janela do ring buffer deixa de ser contígua e a
  entrada sai do domínio do treino. Vale descartar a inferência e esperar encher de
  novo.

In [ ]:
# As constantes saem de MEDIA/DESVIO, calculados no split de treino na secao 2.
# Nao sao transcritas a mao: digitar foi exatamente o que deixou este arquivo
# desatualizado quando os csv foram regerados em unidade fisica.
normalizacao = {
    "janela_horas": JANELA,
    "ordem_das_entradas": NOMES,
    "observacao_janela": (
        f"cada inferencia recebe {JANELA} amostras consecutivas de 1 h, da mais "
        "antiga para a mais recente, cada uma com as features na ordem de 'features'. "
        "O alvo previsto e' o PCA 1 h DEPOIS da amostra mais recente."
    ),
    "features": list(FEATURES),
    "media": {c: float(MEDIA[c]) for c in FEATURES},
    "desvio": {c: float(DESVIO[c]) for c in FEATURES},
    "formula": "x_normalizado = (x_bruto - media) / desvio",
    "fonte": "utils/treino_regressao.ipynb (ajustado somente no split de treino)",
    "alvo": {
        "nome": ALVO,
        "unidade": "kW CA",
        "horizonte": "t+1 (1 hora a frente)",
        "min": PCA_MIN,
        "max": PCA_MAX,
        "observacao": "a saida do modelo ja esta em kW; nao ha transformacao inversa",
    },
}

caminho_json = SAIDA / "normalizacao.json"
caminho_json.write_text(json.dumps(normalizacao, indent=2, ensure_ascii=False), encoding="utf-8")
print(f"{caminho_json.name} gravado")
print(caminho_json.read_text(encoding="utf-8"))

## 11. Resumo

O que ficou em `data/modelo_exportado/`:

| arquivo | para que serve |
|---|---|
| `modelo_edge_impulse.zip` | **é este que sobe no Edge Impulse**, em *Dashboard > Bring your own model*. SavedModel com entrada `(1, 12)` e saída `(1, 1)` |
| `dados_representativos.npy` | amostra da validação para a quantização int8, no formato `(n, 1, 12)` |
| `modelo.tflite` | conversão local, mesmo grafo. Serve para conferir tamanho antes de subir |
| `normalizacao.json` | janela, ordem das entradas e as constantes média/desvio, para o firmware normalizar antes de inferir |

No Edge Impulse, na hora do upload: tipo de entrada **Other**, tipo de saída
**Regression**, e o `.npy` no campo de dados representativos se for quantizar. Depois de
quantizar, **reavalie** — a quantização é com perda e é comum o MAE subir.

### O que mudou nesta versão

- o alvo passou de `PCA(t)` para `PCA(t+1)` — horizonte de 1 hora
- a entrada passou de 6 para 12 valores: `JANELA = 2` horas × 6 features
- a normalização passou a ser feita **aqui**, ajustada só no treino, e o
  `normalizacao.json` é gerado a partir dela em vez de transcrito

O ganho medido da janela, no split de teste e com média de 3 sementes:
**MAE 0,1930 → 0,1496 kW** (−22%). `PCA` defasado como entrada foi testado e não ajuda;
`JANELA` acima de 2 também não.